In [1]:
from pathlib import Path
import pandas as pd
from form990_parser import Form990Parser

%load_ext autoreload
%autoreload 2

In [2]:
file_lookup = pd.read_csv('example_usecase_lookup.csv')
file_lookup.sample(5)

,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990,IRS990Size,IRS990ScheduleA,...,ExplanOfLegisPoliticalActvts,ExplanOfLegisPoliticalActvtsSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize,BorrowedFundsElection,BorrowedFundsElectionSize,TaxUnderSection511Statement,TaxUnderSection511StatementSize
901,A 501c3 filed a 990 containing IRS990ScheduleF...,431634280,CHARITIES AID FOUNDATION AMERICA,501c3,xml\2026\2026_2A\202610489349300906_public.xml,2024,990,1.0,299.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0
645,A 501c3 filed a 990EZ containing IRS990Schedul...,815484985,PEARLS FOR PUPS CO,501c3,xml\2022\2022_01A\202210609349201221_public.xml,2021,990EZ,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
990,A 501c3 filed a 990EZ containing IRS990Schedul...,364754978,ST LOUIS FERAL CAT OUTREACH INC,501c3,xml\2025\2025_12A\202503259349200000_public.xml,2024,990EZ,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
63,A 501c3 filed a 990PF containing AppliedToPrio...,743015612,NATURAL BRIDGE WILDLIFE HERITAGE,501c3,xml\2019\2019_03A\201912839349100211_public.xml,2018,990PF,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
322,A 501c3 filed a 990 containing IRS990ScheduleO...,364724966,Northwestern Memorial HealthCare Group,501c3,xml\2021\2021_01A\202111959349300416_public.xml,2019,990,1.0,459.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
for _ in list(range(2016, 2025)):
    print(_, ",")

2016 ,
2017 ,
2018 ,
2019 ,
2020 ,
2021 ,
2022 ,
2023 ,
2024 ,


In [4]:
schedules = [
    'IRS990ScheduleC',
    'IRS990ScheduleI'
]

df = file_lookup.loc[
    (file_lookup.loc[:, schedules[0]] == 1) | (file_lookup.loc[:, schedules[1]] == 1)
]

tax_years = [
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
]

df = df.loc[df.loc[:, 'TaxYr'].isin(tax_years)]

return_types = [
    '990EZ',
    '990',
    '990PF',
    '990-T'
]

if len(return_types) < 4:
    df = df.loc[df.loc[:, 'ReturnTypeCd'].isin(return_types)]
else:
    pass

org_types = [
    '501c3',
    '501c4',
    '527'
]

if len(org_types) < 3:
    df = df.loc[df.loc[:, 'OrgType'].isin(org_types)]
else:
    pass

In [5]:
import time

In [6]:
header = pd.DataFrame()
schedule_content_accum = {
    'IRS990ScheduleC': {
        'Section527PoliticalOrgGrp': pd.DataFrame(),
        'AvgLobbyingNontaxableAmountGrp': pd.DataFrame(),
        'AvgTotalLobbyingExpendGrp': pd.DataFrame(),
        'AvgGrassrootsNontaxableGrp': pd.DataFrame(),
        'AvgGrassrootsLobbyingExpendGrp': pd.DataFrame(),
        'SupplementalInformationDetail': pd.DataFrame(),
        'Flat Content': pd.DataFrame()
    },
    'IRS990ScheduleI': {
        'RecipientTable': pd.DataFrame(),
        'GrantsOtherAsstToIndivInUSGrp': pd.DataFrame(),
        'SupplementalInformationDetail': pd.DataFrame(),
        'Flat Content': pd.DataFrame()
    }
}

In [ ]:
total_parse_time = 0
flat_content = dict[tuple[int, str], str]
nested_content = dict[str, list[dict[str, str | dict]]]

def flat_content_formatting(_flat_content_key: tuple[int, str], _flat_content_val: str) -> tuple[str, str]:
    return _flat_content_key[1], _flat_content_val

def nested_content_formatting(_nested_content_dict: dict):
    return _nested_content_dict['Nested Content']

for i, file in enumerate(df.file.unique()):
    example_file_path = Path(file)
    prev_time = time.time()



    parser = Form990Parser(example_file_path)

    full_formatter = parser.read_in_formatter('IRS_Form_990_Formatting.json')
    
    header_formatter = full_formatter['Header']
    curr_header = parser.parse_header(
        _header_hierarchy_dict = header_formatter
    )
    header = pd.concat([header, curr_header], axis=0, ignore_index=True)
    
    curr_ein = curr_header.loc[:, 'EIN'].values[0]
    curr_tax_year = curr_header.loc[:, 'TaxYear'].values[0]

    for schedule in schedules:
        # No need to do parsing when our reference file has already checked if it's present
        if len(df.loc[(df.file == file) & (df.loc[:, schedule] == 1)]) == 0:
            continue

        schedule_formatter = full_formatter[schedule]['Content']

        schedule_flat_targets = parser.create_xml_target_mapping_dict(
            _target_type=flat_content,
            _format_mappings_func=flat_content_formatting,
            _dict=schedule_formatter
        )

        # Nested targets
        schedule_nested_targets = parser.create_xml_target_mapping_list(
            _target_type=nested_content,
            _format_mappings_func=nested_content_formatting,
            _dict=schedule_formatter
        )

        schedule_content, missed_content = parser.parse_schedule(
            _ein=curr_ein,
            _tax_year=curr_tax_year,
            _schedule_root=parser.find_element(schedule),
            _flat_targets=schedule_flat_targets,
            _nested_targets=schedule_nested_targets
        )

        for content_key, content_df in schedule_content.items():
            schedule_content_accum[schedule][content_key] = pd.concat(
                [
                    schedule_content_accum[schedule][content_key],
                    content_df
                ],
                axis=0,
                ignore_index=True
            )
        if len(missed_content) > 0:
            print(file)
            print("\t", schedule)
            print("\t", missed_content)



    final_time = time.time()
    total_parse_time += final_time - prev_time
total_parse_time / (i+1)

0.38011427845541884

In [8]:
header

,PreparerEIN,PreparerName,PreparerAddress,PreparerCity,PreparerStateCode,PreparerZipCode,EIN,BusinessName,BusinessNameControlTxt,Phone,...,PreparerPersonName,PreparerPersonPTIN,PreparerPersonPhone,TaxYear,BusinessNameLine2,PreparerAddressLine2,SigningOfficerFirstName,SigningOfficerLastName,SigningOfficerSSN,IRSResponsiblePrtyInfoCurrInd
0,135565207,KPMG LLP,345 PARK AVENUE,NEW YORK,NY,101540102,135562401,120 Broadway,YOUN,6464582400,...,DAVID M HIGHFILL,P01517891,2127589700,2018,NaN,NaN,NaN,NaN,NaN,NaN
1,910189318,MOSS ADAMS LLP,101 SECOND STREET SUITE 900,SAN FRANCISCO,CA,94105,510198509,PO BOX 29903,TIDE,4155616400,...,TRACY S PAGLIA,P00366884,4159561500,2018,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,810421425,5705 GRANT CREEK RD,ROCK,4065234500,...,NaN,NaN,NaN,2018,NaN,NaN,NaN,NaN,NaN,NaN
3,861065772,DELOITTE TAX LLP,555 MISSION STREET,SAN FRANCISCO,CA,94105,311640316,211 MAIN STREET,SCHW,8007466216,...,JOAN S MCMAHON,P00966494,4157834000,2018,NaN,NaN,NaN,NaN,NaN,NaN
4,346565596,ERNST & YOUNG US LLP,155 N WACKER DRIVE,CHICAGO,IL,60606,364724966,541 N Fairbanks Ct Rm 1630,NORT,3129264237,...,JACOB ZEHNDER,P01564049,3128792000,2018,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249,NaN,NaN,NaN,NaN,NaN,NaN,851133849,145 G Street Suite A,COAL,NaN,...,NaN,NaN,NaN,2024,NaN,NaN,NaN,NaN,NaN,NaN
250,850352005,Lawrence Steven Taub Counsellor at Law PC,1000 Cordova Place 618,Santa Fe,NM,87505,850226484,PO Box 2404,INST,5059889800,...,Lawrence S Taub,P00028605,5059843222,2024,NaN,NaN,NaN,NaN,NaN,true
251,941138201,KOKJER PIEROTTI MAIOCCO &DUCK,333 PINE ST 5TH FLOOR,SAN FRANCISCO,CA,94104,943335236,1569 SOLANO AVENUE 703,REGE,5105265993,...,Richard Pierotti,P00314987,4159814224,2024,NaN,NaN,NaN,NaN,NaN,true
252,NaN,NaN,NaN,NaN,NaN,NaN,872296531,2350 Kerner Blvd 250,CALI,4153896800,...,NaN,NaN,NaN,2024,Homelessness and Mental Health Support,NaN,NaN,NaN,NaN,true


In [9]:
print(schedule_content_accum.keys())
schedule_content_accum['IRS990ScheduleI'].keys()

dict_keys(['IRS990ScheduleC', 'IRS990ScheduleI'])


dict_keys(['RecipientTable', 'GrantsOtherAsstToIndivInUSGrp', 'SupplementalInformationDetail', 'Flat Content'])

In [10]:
schedule_content_accum['IRS990ScheduleI']['GrantsOtherAsstToIndivInUSGrp']

,FilerEIN,TaxYear,GrantType,Recipients,CashGrant,NonCashAssistance,ValuationMethodUsedDesc,NonCashAssistanceDesc
0,135562401,2018,ECONOMIC ASSISTANCE,13,17090,0,NaN,NaN
1,510198509,2018,SCHOLARSHIPS,25,118614,NaN,NaN,NaN
2,810421425,2018,SCHOLARSHIP,12,36000,NaN,NaN,NaN
3,364724966,2018,SCHOLARSHIPS,138,294339,NaN,NaN,NaN
4,364724966,2018,EMPLOYEE CRISIS ASSISTANCE,195,290021,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
343,912153073,2024,FELLOWSHIPS,2156,119894213,NaN,NaN,NaN
344,204560286,2024,EMERGING LEADERS FELLOWSHIP STIPEND,9,90000,NaN,NaN,NaN
345,237014089,2024,CHARITABLE CONTRIBUTION,841,1983592,NaN,NaN,NaN
346,237028846,2024,SCHOLARSHIPS,2796,1397771,0,NaN,NaN
